# Sanity check for embeddings.npy

After removing sentences like New York Times Bestseller from the descriptions, lets see if the embedings improved.

In [1]:
import numpy as np
import pandas as pd

In [2]:
OLD_EMBEDDINGD_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/embeddings_old.npy"

BOOK_ID_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/book_ids.npy"
EMBEDDINGD_PATH = "/home/claraoberle/personal-book-recommender/data/embedding/embeddings.npy"
CLEAN_DF_PATH = "/home/claraoberle/personal-book-recommender/data/clean/goodreads_clean.csv"

In [3]:
book_ids = np.load(BOOK_ID_PATH, allow_pickle=True)
embeddings_new = np.load(EMBEDDINGD_PATH, allow_pickle=True)
embeddings_old = np.load(OLD_EMBEDDINGD_PATH)
clean_df = pd.read_csv(CLEAN_DF_PATH)

I will check the top similarity books to several read books and see if the match is reasonable.

In [4]:
def print_top_similarities(book_id, embedding_old, embedding_new, clean_df, book_ids):
    # Find the vector for the query book in both embeddings
    idx = np.where(book_ids == str(book_id))[0][0]

    query_vector_old = embedding_old[idx]
    query_vector_new = embedding_new[idx]

    # Calculate similarity with every other book
    similarities_old = embedding_old @ query_vector_old
    similarities_new = embedding_new @ query_vector_new

    # Make similarity with itself -1
    similarities_old[idx] = -1
    similarities_new[idx] = -1

    # Get top 5 for each embedding
    top_indices_old = np.argsort(similarities_old)[-5:][::-1]
    top_indices_new = np.argsort(similarities_new)[-5:][::-1]

    # Create results tables
    results_old = clean_df[
        clean_df["Book Id"].astype(str).isin(book_ids[top_indices_old])
    ].copy()

    results_new = clean_df[
        clean_df["Book Id"].astype(str).isin(book_ids[top_indices_new])
    ].copy()

    # Add similarity scores
    results_old["similarity"] = [
        similarities_old[np.where(book_ids == str(book_id))[0][0]]
        for book_id in results_old["Book Id"]
    ]

    results_new["similarity"] = [
        similarities_new[np.where(book_ids == str(book_id))[0][0]]
        for book_id in results_new["Book Id"]
    ]

    # Sort
    results_old = results_old[
        ["Book Id", "Title", "similarity"]
    ].sort_values("similarity", ascending=False)

    results_new = results_new[
        ["Book Id", "Title", "similarity"]
    ].sort_values("similarity", ascending=False)

    # Print both
    print("=== OLD EMBEDDING ===")
    print(results_old.to_string(index=False))

    print("\n=== NEW EMBEDDING ===")
    print(results_new.to_string(index=False))

In [5]:
the_hunger_games_id = "2767052"
el_señor_de_los_anillos = "222947"
paper_towns = "6442769"
the_selection = "10507293"

print_top_similarities(the_hunger_games_id, embeddings_old, embeddings_new, clean_df, book_ids)

=== OLD EMBEDDING ===
  Book Id                                                                     Title  similarity
  6148028                                      Catching Fire (The Hunger Games, #2)    0.693397
183526560                 Dune (edición especial película) (Las crónicas de Dune 1)    0.631514
  1908374                                                                     Aloma    0.604867
        6                    Harry Potter and the Goblet of Fire (Harry Potter, #4)    0.548952
 15749186 To All the Boys I've Loved Before (To All the Boys I've Loved Before, #1)    0.547155

=== NEW EMBEDDING ===
  Book Id                                                                     Title  similarity
  6148028                                      Catching Fire (The Hunger Games, #2)    0.693397
183526560                 Dune (edición especial película) (Las crónicas de Dune 1)    0.631514
  1908374                                                                     Aloma    0.60

In [6]:
print_top_similarities(el_señor_de_los_anillos, embeddings_old, embeddings_new, clean_df, book_ids)

=== OLD EMBEDDING ===
 Book Id                                                        Title  similarity
 7183286                   El imperio final (Nacidos de la bruma, #1)    0.614532
10664113            A Dance with Dragons (A Song of Ice and Fire, #5)    0.604951
       1    Harry Potter and the Half-Blood Prince (Harry Potter, #6)    0.559236
       2 Harry Potter and the Order of the Phoenix (Harry Potter, #5)    0.547279
 7235533                The Way of Kings (The Stormlight Archive, #1)    0.540361

=== NEW EMBEDDING ===
 Book Id                                                            Title  similarity
 7183286                       El imperio final (Nacidos de la bruma, #1)    0.614532
18635622 A Knight of the Seven Kingdoms (The Tales of Dunk and Egg, #1-3)    0.568910
       1        Harry Potter and the Half-Blood Prince (Harry Potter, #6)    0.559236
39943621                           Fire & Blood (A Targaryen History, #1)    0.558403
       2     Harry Potter and the

In [7]:
print_top_similarities(paper_towns, embeddings_old, embeddings_new, clean_df, book_ids)

=== OLD EMBEDDING ===
 Book Id                                           Title  similarity
54493401                               Project Hail Mary    0.577395
   68429            The Well of Ascension (Mistborn, #2)    0.528333
36307634               King of Scars (King of Scars, #1)    0.510153
54189398 The Spanish Love Deception (Love Deception, #1)    0.502625
   11334                                 Song of Solomon    0.501691

=== NEW EMBEDDING ===
  Book Id                             Title  similarity
 16248068     The Elite (The Selection, #2)    0.458202
115559438              Parable of the Sower    0.451372
219848315           The Emperor of Gladness    0.449697
   102806                 Olvidado rey Gudú    0.449224
   500400 El amor en los tiempos del cólera    0.448677


In [8]:
print_top_similarities(the_selection, embeddings_old, embeddings_new, clean_df, book_ids)

=== OLD EMBEDDING ===
  Book Id                                                    Title  similarity
 16248068                            The Elite (The Selection, #2)    0.579982
 21415180                                        Regreso a tu piel    0.490199
 18126198 Four: A Divergent Story Collection (Divergent, #0.1-0.4)    0.462148
 10429045                              Shatter Me (Shatter Me, #1)    0.449603
115559438                                     Parable of the Sower    0.402897

=== NEW EMBEDDING ===
 Book Id                                                    Title  similarity
16248068                            The Elite (The Selection, #2)    0.579982
21415180                                        Regreso a tu piel    0.490199
45023611                              The Roommate (Shameless #1)    0.465431
18126198 Four: A Divergent Story Collection (Divergent, #0.1-0.4)    0.462148
10429045                              Shatter Me (Shatter Me, #1)    0.449603
